# BiasPruner on HAM10000

Attach HAM10000 in Kaggle and enable a GPU accelerator. This notebook configures the paper's three-task experiment; training is an explicit cell after the sanity checks.

In [ ]:
from pathlib import Path
import sys
PROJECT = Path('/kaggle/working/biaspruner-ham10000')
if PROJECT.exists():
    sys.path.insert(0, str(PROJECT))
else:
    PROJECT = Path.cwd()
    sys.path.insert(0, str(PROJECT))
DATA_ROOT = '/kaggle/input/ham10000'  # edit to the attached dataset's Kaggle slug
METADATA_CSV = None
IMAGE_DIRS = ()
OUTPUT_DIR = '/kaggle/working/biaspruner-results'
print('Project:', PROJECT, '| HAM root:', DATA_ROOT)

In [ ]:
from config import Config
from dataset import HAM_CLASSES, task_class_groups
cfg = Config(data_root=DATA_ROOT, metadata_csv=METADATA_CSV, image_dirs=IMAGE_DIRS, output_dir=OUTPUT_DIR)
cfg.validate()
print('Classes:', HAM_CLASSES)
print('Task groups:', task_class_groups(cfg.task_order))
print('GCE epochs:', cfg.gce_epochs, '| WCE epochs:', cfg.finetune_epochs, '| q/tau/gamma:', cfg.q, cfg.tau, cfg.gamma)
print('Random task orders:', cfg.num_task_orders, '| fixed alpha:', cfg.alpha)
print('GPU available:', __import__('torch').cuda.is_available())

In [ ]:
# Dataset-only check. Does not start model training.
from dataset import load_ham_frame, split_lesions
frame = load_ham_frame(cfg.data_root, cfg.metadata_csv, cfg.image_dirs)
splits = split_lesions(frame, cfg.seed, cfg.train_fraction, cfg.val_fraction)
print({name: len(part) for name, part in splits.items()})
print('Age groups (-1=missing, 0=<60, 1=>=60):', frame.age_group.value_counts().to_dict())

In [ ]:
# Sanity checks: attached HAM data plus lightweight synthetic pipeline checks.
from sanity_check import run_sanity
run_sanity(DATA_ROOT, METADATA_CSV, IMAGE_DIRS)

In [ ]:
# Full training: paper settings (200 GCE max + 20 WCE epochs); default is 3 random orders.
# For debugging, set cfg.num_task_orders = 1 before running this cell.
from train import run_experiments
run_experiments(cfg)

In [ ]:
# Each order saves evaluation.json; summary.json reports mean and standard deviation.
import json
print(json.loads((Path(OUTPUT_DIR) / 'summary.json').read_text())['summary'])